# Soil Moisture Forecasting with U-Net

This notebook implements a **U-Net based deep learning model** to forecast soil moisture from multi-feature geospatial time-series data.

## Overview
The pipeline consists of the following major stages:

| Stage | Description |
|---|---|
| **1. Configuration** | Define all hyperparameters and file paths in one place |
| **2. Data Loading** | Load yearly `.npy` feature arrays and stack them into a multi-channel format |
| **3. Normalisation** | Compute per-feature mean & std from training years; apply z-score normalisation |
| **4. Dataset & DataLoaders** | Build PyTorch `Dataset` with sliding-window samples and wrap in `DataLoader` |
| **5. U-Net Model** | Define encoder–bottleneck–decoder architecture with skip connections |
| **6. Training** | Train with MSE loss and Adam optimiser; save checkpoint & loss curve |
| **7. Validation** | Run inference, compute masked metrics (MSE, Variance-Reduced MSE, R²) |
| **8. Saving Outputs** | Persist predictions, targets, mask, metrics and sample visualisation |

---
**Input features (6 channels):** Evaporation, Rainfall, Soil Moisture, Surface Solar Radiation, U-wind component, V-wind component  
**Target:** Soil moisture for the next `forecast_horizon` days  
**Spatial mask:** `WB_mask.npy` — keeps only valid land pixels in evaluation

---
## 1. Imports

In [18]:
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

---
## 2. Configuration

All hyperparameters and path definitions live in the `Config` dataclass.  
Adjust values here — no need to hunt through the code.

| Parameter | Default | Meaning |
|---|---|---|
| `train_years` | 2001–2018 | Years used for training |
| `val_years` | 2019–2020 | Years used for validation |
| `input_days` | 4 | Number of past days fed as input |
| `forecast_horizon` | 3 | Number of future days to predict |
| `soil_moisture_feature_index` | 2 | Index of soil moisture in the feature stack |
| `batch_size` | 32 | Mini-batch size |
| `num_epochs` | 20 | Training epochs |
| `learning_rate` | 1e-3 | Adam learning rate |
| `base_filters` | 32 | Base channel width for the U-Net |
| `eps` | 1e-6 | Small constant to avoid division by zero |

In [19]:
@dataclass
class Config:
    project_root: Path = Path.cwd()
    train_years: Tuple[int, ...] = tuple(range(2001, 2019))
    val_years: Tuple[int, ...] = (2019, 2020)
    input_days: int = 4
    forecast_horizon: int = 3
    soil_moisture_feature_index: int = 2
    batch_size: int = 32
    num_epochs: int = 20
    learning_rate: float = 1e-3
    base_filters: int = 32
    num_workers: int = 0
    eps: float = 1e-6
    save_dir_name: str = "outputs"
    checkpoint_name: str = "soil_moisture_unet_checkpoint.pth"
    metrics_name: str = "validation_metrics.json"
    prediction_tag: str = "validation"

    @property
    def feature_dirs(self) -> Dict[str, Path]:
        """Mapping from feature name to its data directory."""
        return {
            "evaporation": self.project_root / "yearly_data_evaporation",
            "rainfall": self.project_root / "yearly_data_rainfall",
            "soil_moisture": self.project_root / "yearly_data_soil_moisture",
            "surface_solar_rad": self.project_root / "yearly_data_surface_solar_rad",
            "u_component": self.project_root / "yearly_data_u_component",
            "v_component": self.project_root / "yearly_data_v_component",
        }

    @property
    def mask_path(self) -> Path:
        """Path to the spatial land/water binary mask."""
        return self.project_root / "WB_mask.npy"

    @property
    def save_dir(self) -> Path:
        """Directory where all outputs (checkpoints, metrics, plots) are saved."""
        return self.project_root / self.save_dir_name

---
## 3. Utility Functions

Small helpers used throughout the pipeline.

In [20]:
def seed_everything(seed: int = 42) -> None:
    """Fix random seeds for reproducibility across NumPy, PyTorch CPU and CUDA."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    """Return GPU if available, otherwise CPU."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def ensure_output_dir(path: Path) -> None:
    """Create the output directory (and any parents) if it doesn't already exist."""
    path.mkdir(parents=True, exist_ok=True)

---
## 4. Data Loading

### File format
Each feature is stored as a collection of yearly `.npy` files named `yearly_data_<YEAR>.npy`.
- Shape per file: `(days, H, W)` — days in the year × spatial grid height × width.
- All six features are loaded for a given year, trimmed to the same number of days (in case of small mismatches), and stacked into a single array of shape `(days, 6, H, W)`.

In [21]:
def load_feature_file(directory: Path, year: int) -> np.ndarray:
    """
    Load a single feature's yearly .npy file.

    Returns an array of shape (days, H, W) as float32.
    Raises FileNotFoundError if the file is missing, or ValueError for bad shapes.
    """
    file_path = directory / f"yearly_data_{year}.npy"
    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {file_path}")
    array = np.load(file_path).astype(np.float32)
    if array.ndim == 2:
        array = array[np.newaxis, ...]          # add a leading day dimension
    if array.ndim != 3:
        raise ValueError(f"Expected 3D array in {file_path}, got shape {array.shape}")
    return array


def load_year_stack(config: Config, year: int) -> np.ndarray:
    """
    Load and stack all 6 features for a given year.

    Steps:
      1. Load each feature file  →  list of (days_i, H, W) arrays.
      2. Trim all to the minimum day count  →  (min_days, H, W) each.
      3. Stack along axis=1  →  (min_days, 6, H, W).

    Returns float32 array of shape (days, num_features, H, W).
    """
    feature_arrays = [
        load_feature_file(directory, year)
        for directory in config.feature_dirs.values()
    ]
    min_days = min(array.shape[0] for array in feature_arrays)
    trimmed = [array[:min_days] for array in feature_arrays]
    stacked = np.stack(trimmed, axis=1)             # (days, 6, H, W)
    return stacked.astype(np.float32)

---
## 5. Feature Normalisation

Per-feature **z-score** (mean & std) statistics are computed **only from the training years** to avoid data leakage.  
The computation is done incrementally (year by year) to avoid loading everything into RAM at once.

$$\mu_f = \frac{\sum_{\text{all pixels}} x_f}{N}, \quad \sigma_f = \sqrt{\frac{\sum x_f^2}{N} - \mu_f^2}$$

In [22]:
def compute_feature_stats(
    config: Config, years: Sequence[int]
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute per-feature mean and std across all pixels of all given years.

    Uses Welford-style incremental summation to keep memory usage low.

    Returns:
        mean  : float32 array of shape (num_features,)
        std   : float32 array of shape (num_features,)
    """
    sums = None
    sq_sums = None
    pixel_count = 0

    for year in tqdm(years, desc="Computing feature stats"):
        year_data = load_year_stack(config, year)   # (days, F, H, W)
        if sums is None:
            sums = np.zeros(year_data.shape[1], dtype=np.float64)
            sq_sums = np.zeros(year_data.shape[1], dtype=np.float64)
        sums += year_data.sum(axis=(0, 2, 3))       # sum over days, H, W
        sq_sums += np.square(year_data, dtype=np.float64).sum(axis=(0, 2, 3))
        pixel_count += year_data.shape[0] * year_data.shape[2] * year_data.shape[3]

    if sums is None or sq_sums is None or pixel_count == 0:
        raise ValueError("No training data found to compute statistics.")

    mean = sums / pixel_count
    variance = np.maximum((sq_sums / pixel_count) - np.square(mean), config.eps)
    std = np.sqrt(variance)
    return mean.astype(np.float32), std.astype(np.float32)

---
## 6. PyTorch Dataset

### Sliding-window sampling

For each year and each valid `start_day`, one sample is:
- **Input** : days `[start_day, start_day + input_days)` — all 6 features, flattened to `(6 × input_days, H, W)`
- **Target** : days `[start_day + input_days, start_day + input_days + forecast_horizon)` — soil moisture only, shape `(forecast_horizon, H, W)`

The data is normalised **before** building samples to avoid redundant computation.

In [23]:
class SoilMoistureDataset(Dataset):
    """
    Sliding-window dataset for multi-step soil moisture forecasting.

    Each sample consists of:
      - input_tensor : (num_features * input_days, H, W) float32
      - target_window: (forecast_horizon, H, W) float32  [soil moisture only]
      - metadata dict : year, input_start_day, target_start_day
    """

    def __init__(
        self,
        config: Config,
        years: Sequence[int],
        feature_mean: np.ndarray,
        feature_std: np.ndarray,
    ) -> None:
        self.config = config
        # Reshape stats for broadcasting: (1, F, 1, 1)
        self.feature_mean = feature_mean.reshape(1, -1, 1, 1)
        self.feature_std = feature_std.reshape(1, -1, 1, 1)
        self.samples: List[Tuple[np.ndarray, int, int]] = []

        for year in tqdm(years, desc="Loading dataset"):
            year_data = load_year_stack(config, year)           # (days, F, H, W)
            year_data = (year_data - self.feature_mean) / self.feature_std  # z-score
            n_days = year_data.shape[0]
            # Number of valid starting positions for the sliding window
            max_start = n_days - config.input_days - config.forecast_horizon + 1
            if max_start <= 0:
                continue
            for start_day in range(max_start):
                self.samples.append((year_data, year, start_day))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        year_data, year, start_day = self.samples[index]

        # Input window: (input_days, F, H, W)  →  transpose  →  (F, input_days, H, W)
        #               →  reshape  →  (F * input_days, H, W)
        input_window = year_data[start_day : start_day + self.config.input_days]
        input_tensor = np.transpose(input_window, (1, 0, 2, 3)).reshape(
            -1, input_window.shape[2], input_window.shape[3]
        )

        # Target window: soil moisture channel only, (forecast_horizon, H, W)
        target_window = year_data[
            start_day + self.config.input_days
            : start_day + self.config.input_days + self.config.forecast_horizon,
            self.config.soil_moisture_feature_index,
            :,
            :,
        ]

        metadata = {
            "year": year,
            "input_start_day": start_day,
            "target_start_day": start_day + self.config.input_days,
        }

        return (
            torch.tensor(input_tensor, dtype=torch.float32),
            torch.tensor(target_window, dtype=torch.float32),
            metadata,
        )

---
## 7. U-Net Architecture

### Design
The model uses a standard **encoder–bottleneck–decoder** structure with **skip connections** (concatenation).

```
Input (F×D, H, W)
    │
  enc1 (32)  ──────────────────────────────────┐
    │ pool                                      │
  enc2 (64)  ────────────────────────┐          │
    │ pool                           │          │
  enc3 (128) ──────────┐             │          │
    │ pool             │             │          │
  bottleneck (256)     │             │          │
    │ up3              │             │          │
  dec3 (128) ←─── cat(enc3) ────────┘          │
    │ up2              │                        │
  dec2 (64)  ←─── cat(enc2) ───────────────────┘
    │ up1
  dec1 (32)  ←─── cat(enc1)
    │
  head (forecast_horizon, H, W)
```

Each `DoubleConv` block applies two `Conv2d → BN → ReLU` operations.  
`_match_size` handles any minor spatial mismatches between encoder and decoder paths caused by max-pooling on odd-sized inputs.

In [24]:
class DoubleConv(nn.Module):
    """
    Two consecutive (Conv2d → BatchNorm2d → ReLU) blocks.

    This is the fundamental building block of the U-Net encoder and decoder.
    """

    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

In [25]:
class UNet(nn.Module):
    """
    U-Net with 3 encoder levels, a bottleneck, and 3 decoder levels.

    Args:
        in_channels   : num_features × input_days  (e.g. 6 × 4 = 24)
        out_channels  : forecast_horizon           (e.g. 3)
        base_filters  : channel width at first encoder level (default 32)
    """

    def __init__(self, in_channels: int, out_channels: int, base_filters: int = 32) -> None:
        super().__init__()
        filters = [base_filters, base_filters * 2, base_filters * 4, base_filters * 8]
        # [32, 64, 128, 256]

        # ── Encoder ──────────────────────────────────────────────────────────
        self.enc1 = DoubleConv(in_channels, filters[0])
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(filters[0], filters[1])
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(filters[1], filters[2])
        self.pool3 = nn.MaxPool2d(2)

        # ── Bottleneck ───────────────────────────────────────────────────────
        self.bottleneck = DoubleConv(filters[2], filters[3])

        # ── Decoder ──────────────────────────────────────────────────────────
        self.up3 = nn.ConvTranspose2d(filters[3], filters[2], kernel_size=2, stride=2)
        self.dec3 = DoubleConv(filters[2] * 2, filters[2])   # *2 because of skip concat
        self.up2 = nn.ConvTranspose2d(filters[2], filters[1], kernel_size=2, stride=2)
        self.dec2 = DoubleConv(filters[1] * 2, filters[1])
        self.up1 = nn.ConvTranspose2d(filters[1], filters[0], kernel_size=2, stride=2)
        self.dec1 = DoubleConv(filters[0] * 2, filters[0])

        # ── Output head ──────────────────────────────────────────────────────
        self.head = nn.Conv2d(filters[0], out_channels, kernel_size=1)

    @staticmethod
    def _match_size(
        source: torch.Tensor, reference: torch.Tensor
    ) -> torch.Tensor:
        """
        Bilinearly resize `source` to match the spatial dimensions of `reference`.
        Needed when max-pooling on odd spatial sizes causes off-by-one mismatches.
        """
        if source.shape[-2:] != reference.shape[-2:]:
            source = nn.functional.interpolate(
                source,
                size=reference.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        return source

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder path
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        enc3 = self.enc3(self.pool2(enc2))
        bottleneck = self.bottleneck(self.pool3(enc3))

        # Decoder path with skip connections
        dec3 = self.up3(bottleneck)
        dec3 = self._match_size(dec3, enc3)
        dec3 = self.dec3(torch.cat([dec3, enc3], dim=1))

        dec2 = self.up2(dec3)
        dec2 = self._match_size(dec2, enc2)
        dec2 = self.dec2(torch.cat([dec2, enc2], dim=1))

        dec1 = self.up1(dec2)
        dec1 = self._match_size(dec1, enc1)
        dec1 = self.dec1(torch.cat([dec1, enc1], dim=1))

        return self.head(dec1)

---
## 8. Training Loop

One epoch: iterate over batches, forward pass, compute MSE loss, backprop, update weights.  
Returns the **average MSE loss** for the epoch.

In [26]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
) -> float:
    """
    Run one full training epoch.

    Returns:
        Average MSE loss per sample across the entire dataset.
    """
    model.train()
    running_loss = 0.0

    for inputs, targets, _ in tqdm(loader, desc="Training", leave=False):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)   # slightly faster than zero_grad()
        predictions = model(inputs)
        loss = loss_fn(predictions, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)  # accumulate un-averaged loss

    return running_loss / max(len(loader.dataset), 1)

---
## 9. Validation & Metrics

### Denormalisation
Predictions and targets are stored in normalised space. Before computing metrics, they are converted back to original units:
$$\hat{y} = y_{\text{norm}} \times \sigma_{\text{SM}} + \mu_{\text{SM}}$$

### Metrics (per horizon step + overall)
| Metric | Formula | Interpretation |
|---|---|---|
| **MSE** | $\frac{1}{N}\sum(\hat{y}-y)^2$ | Raw error magnitude |
| **Variance-Reduced MSE** | $1 - \frac{\text{MSE}}{\text{Var}(y)}$ | Fraction of variance explained; 1 = perfect |
| **R²** | Pearson $R^2$ | Overall fit quality |

In [27]:
def denormalize_soil_moisture(
    array: np.ndarray,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
    soil_moisture_feature_index: int,
) -> np.ndarray:
    """Reverse z-score normalisation for the soil moisture feature only."""
    sm_mean = float(feature_mean[soil_moisture_feature_index])
    sm_std = float(feature_std[soil_moisture_feature_index])
    return (array * sm_std) + sm_mean


def compute_masked_metrics(
    predictions_norm: np.ndarray,
    targets_norm: np.ndarray,
    mask: np.ndarray,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
    soil_moisture_feature_index: int,
    eps: float,
) -> Dict[str, Dict[str, float]]:
    """
    Compute MSE, variance-reduced MSE, and R² for each forecast horizon step
    plus an overall summary — evaluated only on masked (valid land) pixels.

    Args:
        predictions_norm : (N, forecast_horizon, H, W) normalised predictions
        targets_norm     : (N, forecast_horizon, H, W) normalised targets
        mask             : (H, W) binary array; 1 = valid pixel

    Returns:
        Dict keyed by 't+1', 't+2', ..., 'overall',
        each containing {'MSE': float, 'variance_reduced_MSE': float, 'R2': float}.
    """
    predictions = denormalize_soil_moisture(
        predictions_norm, feature_mean, feature_std, soil_moisture_feature_index
    )
    targets = denormalize_soil_moisture(
        targets_norm, feature_mean, feature_std, soil_moisture_feature_index
    )

    mask_bool = mask.astype(bool)
    metrics: Dict[str, Dict[str, float]] = {}

    # Per-horizon metrics
    for horizon in range(predictions.shape[1]):
        pred_flat = predictions[:, horizon][:, mask_bool].reshape(-1)
        target_flat = targets[:, horizon][:, mask_bool].reshape(-1)

        mse = float(np.mean((pred_flat - target_flat) ** 2))
        variance = float(np.var(target_flat))
        variance_reduced_mse = float(1.0 - (mse / (variance + eps)))
        r2 = float(r2_score(target_flat, pred_flat))

        metrics[f"t+{horizon + 1}"] = {
            "MSE": mse,
            "variance_reduced_MSE": variance_reduced_mse,
            "R2": r2,
        }

    # Overall metrics (all horizons combined)
    pred_all = predictions[:, :, :, :][:, :, mask_bool].reshape(-1)
    target_all = targets[:, :, :, :][:, :, mask_bool].reshape(-1)
    overall_mse = float(np.mean((pred_all - target_all) ** 2))
    overall_variance = float(np.var(target_all))
    metrics["overall"] = {
        "MSE": overall_mse,
        "variance_reduced_MSE": float(1.0 - (overall_mse / (overall_variance + eps))),
        "R2": float(r2_score(target_all, pred_all)),
    }

    return metrics


def validate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    mask: np.ndarray,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
    config: Config,
) -> Tuple[Dict[str, Dict[str, float]], np.ndarray, np.ndarray, List[Dict[str, int]]]:
    """
    Run inference on the validation loader and compute evaluation metrics.

    Returns:
        metrics          : per-horizon and overall metric dict
        predictions_norm : (N, forecast_horizon, H, W) normalised predictions
        targets_norm     : (N, forecast_horizon, H, W) normalised targets
        metadata_rows    : list of per-sample dicts (year, start days)
    """
    model.eval()
    all_predictions: List[np.ndarray] = []
    all_targets: List[np.ndarray] = []
    metadata_rows: List[Dict[str, int]] = []

    with torch.no_grad():
        for inputs, targets, metadata in tqdm(loader, desc="Validation", leave=False):
            predictions = model(inputs.to(device)).cpu().numpy()
            all_predictions.append(predictions)
            all_targets.append(targets.numpy())

            batch_size = predictions.shape[0]
            for i in range(batch_size):
                metadata_rows.append(
                    {
                        "year": int(metadata["year"][i]),
                        "input_start_day": int(metadata["input_start_day"][i]),
                        "target_start_day": int(metadata["target_start_day"][i]),
                    }
                )

    predictions_norm = np.concatenate(all_predictions, axis=0)
    targets_norm = np.concatenate(all_targets, axis=0)

    metrics = compute_masked_metrics(
        predictions_norm,
        targets_norm,
        mask,
        feature_mean,
        feature_std,
        config.soil_moisture_feature_index,
        config.eps,
    )
    return metrics, predictions_norm, targets_norm, metadata_rows

---
## 10. Saving Outputs

Functions for persisting model checkpoints, metrics, and masked predictions to disk.

In [28]:
def save_masked_predictions(
    predictions_norm: np.ndarray,
    targets_norm: np.ndarray,
    mask: np.ndarray,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
    config: Config,
    metadata_rows: Sequence[Dict[str, int]],
) -> None:
    """
    Denormalise predictions & targets, apply the spatial mask,
    and save as .npy files along with per-sample metadata JSON.
    """
    predictions = denormalize_soil_moisture(
        predictions_norm, feature_mean, feature_std, config.soil_moisture_feature_index
    )
    targets = denormalize_soil_moisture(
        targets_norm, feature_mean, feature_std, config.soil_moisture_feature_index
    )

    mask_4d = mask[np.newaxis, np.newaxis, :, :].astype(np.float32)
    masked_predictions = predictions * mask_4d
    masked_targets = targets * mask_4d

    np.save(config.save_dir / f"{config.prediction_tag}_predictions_masked.npy", masked_predictions)
    np.save(config.save_dir / f"{config.prediction_tag}_targets_masked.npy", masked_targets)
    np.save(config.save_dir / f"{config.prediction_tag}_mask.npy", mask.astype(np.uint8))
    (config.save_dir / f"{config.prediction_tag}_metadata.json").write_text(
        json.dumps(metadata_rows, indent=2),
        encoding="utf-8",
    )


def save_metrics(metrics: Dict[str, Dict[str, float]], config: Config) -> None:
    """Serialise validation metrics to a JSON file."""
    (config.save_dir / config.metrics_name).write_text(
        json.dumps(metrics, indent=2), encoding="utf-8"
    )


def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
    config: Config,
    epoch: int,
) -> None:
    """
    Save model weights, optimiser state, normalisation stats, and config
    into a single .pth checkpoint file for later inference or fine-tuning.
    """
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "config": {
            **asdict(config),
            "project_root": str(config.project_root),
            "train_years": list(config.train_years),
            "val_years": list(config.val_years),
        },
        "epoch": epoch,
    }
    torch.save(payload, config.save_dir / config.checkpoint_name)

---
## 11. Visualisation

Two diagnostic plots are produced after training:
1. **Training loss curve** — MSE vs. epoch.
2. **Validation sample** — side-by-side spatial maps of target vs. predicted soil moisture for each forecast step, with invalid pixels masked to `NaN`.

In [29]:
def plot_training_curve(training_losses: Sequence[float], config: Config) -> None:
    """Plot and save the per-epoch training MSE loss curve."""
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(training_losses) + 1), training_losses, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Training MSE")
    plt.title("UNet Training Loss")
    plt.tight_layout()
    plt.savefig(config.save_dir / "training_loss_curve.png", dpi=150)
    plt.close()

def plot_validation_curve(val_losses: Sequence[float], config: Config) -> None:
    """Plot and save the per-epoch validation MSE loss curve."""
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(val_losses) + 1), val_losses, marker="o", color="orange")
    plt.xlabel("Epoch")
    plt.ylabel("Validation MSE")
    plt.title("UNet Validation Loss")
    plt.tight_layout()
    plt.savefig(config.save_dir / "validation_loss_curve.png", dpi=150)
    plt.close()


# def plot_validation_sample(
#     predictions_norm: np.ndarray,
#     targets_norm: np.ndarray,
#     mask: np.ndarray,
#     feature_mean: np.ndarray,
#     feature_std: np.ndarray,
#     config: Config,
#     sample_index: int = 0,
# ) -> None:
#     """
#     Plot target vs. predicted soil moisture maps for one validation sample.

#     Creates a (2 × forecast_horizon) grid:
#       Row 0: ground-truth soil moisture maps (t+1, t+2, ...)
#       Row 1: predicted soil moisture maps    (t+1, t+2, ...)
    
#     Out-of-mask pixels are set to NaN and rendered transparently.
#     """
#     predictions = denormalize_soil_moisture(
#         predictions_norm[sample_index : sample_index + 1],
#         feature_mean,
#         feature_std,
#         config.soil_moisture_feature_index,
#     )[0]
#     targets = denormalize_soil_moisture(
#         targets_norm[sample_index : sample_index + 1],
#         feature_mean,
#         feature_std,
#         config.soil_moisture_feature_index,
#     )[0]

#     fig, axes = plt.subplots(2, config.forecast_horizon, figsize=(5 * config.forecast_horizon, 8))
#     if config.forecast_horizon == 1:
#         axes = np.array(axes).reshape(2, 1)

#     for horizon in range(config.forecast_horizon):
#         target_map = np.where(mask, targets[horizon], np.nan)
#         pred_map = np.where(mask, predictions[horizon], np.nan)
#         vmin = np.nanmin([target_map, pred_map])
#         vmax = np.nanmax([target_map, pred_map])

#         axes[0, horizon].imshow(target_map, cmap="Blues", vmin=vmin, vmax=vmax)
#         axes[0, horizon].set_title(f"Target t+{horizon + 1}")
#         axes[0, horizon].axis("off")

#         image = axes[1, horizon].imshow(pred_map, cmap="Blues", vmin=vmin, vmax=vmax)
#         axes[1, horizon].set_title(f"Prediction t+{horizon + 1}")
#         axes[1, horizon].axis("off")

#     plt.colorbar(image, ax=axes.ravel().tolist(), label="Soil moisture")
#     plt.tight_layout()
#     plt.savefig(config.save_dir / "validation_sample_masked.png", dpi=150)
#     plt.close()

In [30]:
def plot_validation_sample(
    predictions_norm: np.ndarray,
    targets_norm: np.ndarray,
    mask: np.ndarray,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
    config: Config,
) -> None:
    # Denormalize all samples at once
    predictions = denormalize_soil_moisture(
        predictions_norm, feature_mean, feature_std, config.soil_moisture_feature_index
    )  # (N, forecast_horizon, H, W)
    targets = denormalize_soil_moisture(
        targets_norm, feature_mean, feature_std, config.soil_moisture_feature_index
    )  # (N, forecast_horizon, H, W)

    mask_bool = mask.astype(bool)

    fig, axes = plt.subplots(2, config.forecast_horizon, figsize=(5 * config.forecast_horizon, 8))
    if config.forecast_horizon == 1:
        axes = np.array(axes).reshape(2, 1)

    for horizon in range(config.forecast_horizon):
        # Compute per-sample MSE over masked pixels for this horizon step
        diff_sq = (predictions[:, horizon] - targets[:, horizon]) ** 2  # (N, H, W)
        per_sample_mse = diff_sq[:, mask_bool].mean(axis=1)             # (N,)
        worst_idx = int(np.argmax(per_sample_mse))                      # sample with max MSE

        target_map = np.where(mask, targets[worst_idx, horizon], np.nan)
        pred_map   = np.where(mask, predictions[worst_idx, horizon], np.nan)
        vmin = np.nanmin([target_map, pred_map])
        vmax = np.nanmax([target_map, pred_map])

        axes[0, horizon].imshow(target_map, cmap="Blues", vmin=vmin, vmax=vmax)
        axes[0, horizon].set_title(f"Target t+{horizon + 1}\n(sample {worst_idx}, MSE={per_sample_mse[worst_idx]:.4f})")
        axes[0, horizon].axis("off")

        image = axes[1, horizon].imshow(pred_map, cmap="Blues", vmin=vmin, vmax=vmax)
        axes[1, horizon].set_title(f"Prediction t+{horizon + 1}")
        axes[1, horizon].axis("off")

    plt.colorbar(image, ax=axes.ravel().tolist(), label="Soil moisture")
    plt.tight_layout()
    plt.savefig(config.save_dir / "validation_sample_masked.png", dpi=150)
    plt.close()

---
## 12. DataLoader Builder & Dataset Summary

Convenience functions to construct both train and validation `DataLoader` objects in one call, and to print a quick summary of available data files.

In [31]:
def build_dataloaders(
    config: Config,
    feature_mean: np.ndarray,
    feature_std: np.ndarray,
) -> Tuple[SoilMoistureDataset, SoilMoistureDataset, DataLoader, DataLoader]:
    """
    Build train and validation datasets and their corresponding DataLoaders.

    Training loader  : shuffled, with pin_memory on CUDA.
    Validation loader: sequential (no shuffle).

    Returns:
        train_dataset, val_dataset, train_loader, val_loader
    """
    train_dataset = SoilMoistureDataset(config, config.train_years, feature_mean, feature_std)
    val_dataset = SoilMoistureDataset(config, config.val_years, feature_mean, feature_std)

    pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=pin_memory,
    )
    return train_dataset, val_dataset, train_loader, val_loader


def print_dataset_summary(config: Config) -> None:
    """Print the number of .npy files found in each feature directory."""
    print("Feature directories:")
    for name, directory in config.feature_dirs.items():
        count = len(list(directory.glob("yearly_data_*.npy")))
        print(f"  {name:<18} -> {count} files")
    print(f"Mask path: {config.mask_path}")

---
## 13. Main Pipeline

The `main()` function ties every stage together in order:

```
seed  →  config  →  mask  →  normalisation stats
  →  datasets & loaders  →  model & optimiser
  →  training loop  →  checkpoint + loss plot
  →  validation  →  metrics + predictions + sample plot
```

In [32]:
def main() -> Dict[str, Dict[str, float]]:
    """
    End-to-end training and evaluation pipeline.

    Returns:
        Validation metrics dict (same structure as compute_masked_metrics output).
    """
    # ── Setup ────────────────────────────────────────────────────────────────
    seed_everything()
    config = Config()
    ensure_output_dir(config.save_dir)
    print_dataset_summary(config)

    device = get_device()
    mask = np.load(config.mask_path).astype(np.float32)
    print(f"Using device: {device}")
    print(f"Mask shape: {mask.shape}, valid pixels: {int(mask.sum())}")

    # ── Normalisation ────────────────────────────────────────────────────────
    feature_mean, feature_std = compute_feature_stats(config, config.train_years)
    print("\nFeature normalisation stats:")
    for idx, feature_name in enumerate(config.feature_dirs.keys()):
        print(f"  {feature_name:<18} mean={feature_mean[idx]:.6f} std={feature_std[idx]:.6f}")

    # ── Datasets & DataLoaders ───────────────────────────────────────────────
    train_dataset, val_dataset, train_loader, val_loader = build_dataloaders(
        config, feature_mean, feature_std
    )
    print(f"\nTraining samples:   {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")

    # ── Model ────────────────────────────────────────────────────────────────
    in_channels = len(config.feature_dirs) * config.input_days   # 6 × 4 = 24
    out_channels = config.forecast_horizon                        # 3
    model = UNet(in_channels, out_channels, base_filters=config.base_filters).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    loss_fn = nn.MSELoss()

    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {parameter_count:,}")

    # ── Training ─────────────────────────────────────────────────────────────
    training_losses: List[float] = []
    best_val_mse = float("inf")
    val_losses: List[float] = []
    for epoch in range(1, config.num_epochs + 1):
        loss_value = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        training_losses.append(loss_value)

        # ADD: validate every epoch and save only if val MSE improves
        val_metrics, _, _, _ = validate(
            model, val_loader, device, mask, feature_mean, feature_std, config
        )
        val_mse = val_metrics["overall"]["MSE"]
        val_losses.append(val_mse)
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            save_checkpoint(model, optimizer, feature_mean, feature_std, config, epoch)
            print(f"Epoch {epoch:02d}/{config.num_epochs} - train MSE: {loss_value:.6f}  val MSE: {val_mse:.6f}  ✓ saved")
        else:
            print(f"Epoch {epoch:02d}/{config.num_epochs} - train MSE: {loss_value:.6f}  val MSE: {val_mse:.6f}")

        # print(f"Epoch {epoch:02d}/{config.num_epochs} - train MSE: {loss_value:.6f}")

    # save_checkpoint(model, optimizer, feature_mean, feature_std, config, config.num_epochs)
    plot_training_curve(training_losses, config)
    plot_validation_curve(val_losses, config)


    # Load best model before final evaluation
    checkpoint = torch.load(config.save_dir / config.checkpoint_name, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    # ── Validation ───────────────────────────────────────────────────────────
    metrics, predictions_norm, targets_norm, metadata_rows = validate(
        model, val_loader, device, mask, feature_mean, feature_std, config
    )
    save_metrics(metrics, config)
    save_masked_predictions(
        predictions_norm, targets_norm, mask, feature_mean, feature_std, config, metadata_rows
    )
    plot_validation_sample(
        predictions_norm, targets_norm, mask, feature_mean, feature_std, config
    )

    # ── Results ──────────────────────────────────────────────────────────────
    print("\nMasked validation metrics:")
    for key, values in metrics.items():
        print(
            f"  {key:<8} "
            f"MSE={values['MSE']:.6f} "
            f"variance_reduced_MSE={values['variance_reduced_MSE']:.6f} "
            f"R2={values['R2']:.6f}"
        )

    print(f"\nSaved outputs to: {config.save_dir}")
    return metrics

---
## 14. Run

Execute the full pipeline by calling `main()`.  
Ensure all input data directories and `WB_mask.npy` are present relative to `project_root` (defaults to the current working directory).

In [33]:
if __name__ == "__main__":
    metrics = main()

Feature directories:
  evaporation        -> 20 files
  rainfall           -> 20 files
  soil_moisture      -> 20 files
  surface_solar_rad  -> 20 files
  u_component        -> 20 files
  v_component        -> 20 files
Mask path: f:\IIT KGP\sem10\MTP 2\WB_mask.npy
Using device: cpu
Mask shape: (33, 25), valid pixels: 120


Computing feature stats: 100%|██████████| 18/18 [00:00<00:00, 34.92it/s]



Feature normalisation stats:
  evaporation        mean=-0.000124 std=0.001000
  rainfall           mean=0.000217 std=0.001000
  soil_moisture      mean=0.251225 std=0.135366
  surface_solar_rad  mean=783518.937500 std=237707.125000
  u_component        mean=0.091564 std=1.847149
  v_component        mean=0.429594 std=1.909163


Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 31.31it/s]



Training samples:   6466
Validation samples: 719
Model parameters: 1,933,123


...

Epoch 18/20 - train MSE: 0.024713  val MSE: 0.001079


Epoch 19/20 - train MSE: 0.025455  val MSE: 0.000753


Epoch 20/20 - train MSE: 0.025010  val MSE: 0.000707


C:\Users\sreya\AppData\Local\Temp\ipykernel_23068\3174666503.py:43: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



Masked validation metrics:
  t+1      MSE=0.000260 variance_reduced_MSE=0.973548 R2=0.973545
  t+2      MSE=0.000692 variance_reduced_MSE=0.929650 R2=0.929643
  t+3      MSE=0.001054 variance_reduced_MSE=0.892973 R2=0.892963
  overall  MSE=0.000669 variance_reduced_MSE=0.932036 R2=0.932029

Saved outputs to: f:\IIT KGP\sem10\MTP 2\outputs
